In [1]:
import sys 
!{sys.executable} -m pip install pandas


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import ast
import pandas as pd
import json
import numpy as np
from neo4j import GraphDatabase
import regex

In [3]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

In [4]:
uri = "bolt://neo4j-gds-apoc-n10s:7687"
username = "neo4j"
password = "neo4jpassword"

In [5]:
driver = GraphDatabase.driver(uri, auth=(username, password))

In [6]:
DATA_DIR = "/app/notebooks/rdb"

### neo4j 초기화
- 아래 코드를 통해 넣을 수 없는 데이터(다른 방법으로 이미 넣어둔 데이터)가 있는 경우 아래 코드는 실행하면 안 됨

In [7]:
with driver.session() as session:
    result = session.run("RETURN 1 AS test")
    print(result.single())

<Record test=1>


In [8]:
with driver.session() as session:
    # 모든 관계와 노드 제거
    result = session.run("MATCH (n) DETACH DELETE n")
    print(result.single())

None


### 데이터 로딩
- 스키마 참고
  - https://confluence.tde.sktelecom.com/pages/viewpage.action?pageId=734203873
- 순서
  - (1) PRODUCT + PRICE
  - (2) VOICE, SMS, DATA, TOPUP, CUSTOMERCONDITION, PRODUCT_GROUP, BENEFITCONDITION, DEDUCTIBLE, RELATION

### PRODUCT + PRICE
- 모바일 요금제 상품
- Label: 요금제

In [9]:
product_table = pd.read_csv(os.path.join(DATA_DIR, "PRODUCT.csv"))
product_table.head(1)

,pmProductID,mappedProductCode,generation,marketingKeyword,productName,productNameInEnglish,lineup,classifiedGroup,productDescription,productSubscriptionCondition,statusOfOperation
0,PA00000001,['NA00007164'],"['LTE generation', '5G generation']","['유심개통', '쓰던폰', 'USIM개통', '비대면', '자급제', '온라인', '데이터100GB 이하', 'T다샵', '다이렉트플랜', '티다이렉트샵', '티다샵', 'wavve할인혜택', '온라인전용요금제', 'T다이렉트전용요금제']",다이렉트5G 38,Direct5G 38,다이렉트플랜,상품 > 기본요금제 > 휴대폰 요금제,월 15GB 데이터를 제공하는 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입 가능한 온라인 전용 무약정 요금제,T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 고객 가입 가능,운영


In [10]:
price_table = pd.read_csv(os.path.join(DATA_DIR, "PRICE.csv"))
price_table.head(1)

,pmProductID,monthlyPrice,monthlyPriceWithoutVAT,monthlyPriceWithSelectableInstallment,billingMethod,netPrice
0,PA00000001,38000,34546,38000,후불,34545


In [11]:
product_table = product_table.merge(price_table, on="pmProductID", how="left")

In [12]:
eng_colnames = [
    'pmProductID', 'mappedProductCode', 'generation', 'marketingKeyword', 
    'productName', 'productNameInEnglish', 'lineup', 'classifiedGroup',
    'productDescription', 'productSubscriptionCondition', 'statusOfOperation',
    'monthlyPrice', 'monthlyPriceWithoutVAT', 'monthlyPriceWithSelectableInstallment', 
    'billingMethod', 'netPrice'
]

kor_colnames = [
    '고유ID', '상품코드매핑', '통신규격', '마케팅키워드',
    '상품명', '영문상품명', '라인업', '상품분류',
    '상품설명', '상품가입조건', '운영상태',
    '월정액', '부가세제외월정액', '선택약정할인포함부가세제외월정액', 
    '청구방법', 'net가격'
]

kor_cols_map = {x:y for x, y in zip(eng_colnames, kor_colnames)}

In [13]:
def type_cast(input_data):
    new_data = None
    if pd.isna(input_data):
        input_data = "[]"
    elif '[' in input_data and ']' in input_data and 'nan' in input_data:
        input_data =  input_data.replace("nan", "")
    
    new_data = ast.literal_eval(input_data)

    return new_data

In [14]:
# 리스트형으로 변환
product_table["generation"] = product_table["generation"].apply(lambda x: type_cast(x))
product_table["marketingKeyword"] = product_table["marketingKeyword"].apply(lambda x: type_cast(x))
product_table["mappedProductCode"] = product_table["mappedProductCode"].apply(lambda x: type_cast(x))

In [15]:
with driver.session() as session:
    for _, row in product_table.iterrows():
        properties = {kor_cols_map[col]: row[col] for col in eng_colnames}
        session.run(
            """
            CREATE (p:요금제 $props)
            """,
            props=properties
        )

### PRICE
- 상품 가격 정보
- Label: 요금
- 관계
  - 요금제 -[:요금정보]-> 요금

In [16]:
# price_table = pd.read_csv(os.path.join(DATA_DIR, "PRICE.csv"))
# price_table.head(1)

In [17]:
# eng_colnames = ['monthlyPrice', 'monthlyPriceWithoutVAT', 'monthlyPriceWithSelectableInstallment', 
#                 'billingMethod', 'netPrice']

# kor_colnames = ['월정액', '부가세제외월정액', '선택약정할인포함부가세제외월정액', 
#                 '청구방법', 'net가격']

# kor_cols_map = {x:y for x, y in zip(eng_colnames, kor_colnames)}

In [89]:
# with driver.session() as session:
#     for _, row in price_table.iterrows():
#         properties = {kor_cols_map[col]: row[col] for col in eng_colnames}
#         session.run(
#             """
#             MATCH (p:요금제 {고유ID: $product_id})
#             MERGE (price:요금 {월정액: $월정액, 부가세제외월정액: $부가세제외월정액, 
#                                선택약정할인포함부가세제외월정액: $선택약정할인포함부가세제외월정액, 
#                                청구방법: $청구방법, net가격: $net가격})
#             MERGE (p)-[:요금정보]->(price)
#             """,
#             product_id=row['pmProductID'],
#             월정액=properties['월정액'],
#             부가세제외월정액=properties['부가세제외월정액'],
#             선택약정할인포함부가세제외월정액=properties['선택약정할인포함부가세제외월정액'],
#             청구방법=properties['청구방법'],
#             net가격=properties['net가격']
#         )

### VOICE
- 음성 제공량 관련 정보
- Label: 음성통화
- 관계
  - 요금제 -[:제공]-> 음성통화

In [18]:
voice_table = pd.read_csv(os.path.join(DATA_DIR, "VOICE.csv"))
voice_table.head(1)

,pmProductID,includedVoiceCall,includedVideoOrValueAddedCall,includedVoiceCallTospecifiedNumbers,refillAmount,refillRange
0,PA00000001,99999,300,NaN,20%,"['영상통화', '부가통화']"


In [19]:
eng_colnames = [
    'includedVoiceCall',
    'includedVideoOrValueAddedCall',
    'includedVoiceCallTospecifiedNumbers',
    'refillAmountRatio',
    'refillRange'
]

kor_colnames = [
    '음성통화제공량',
    '영상및부가통화제공량',
    '지정번호통화제공량',
    '리필비율한도',
    '리필대상'
]

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

In [20]:
# 리스트형으로 변환
voice_table["refillRange"] = voice_table["refillRange"].apply(lambda x: type_cast(x))

# refillAmount 정규화
voice_table["refillAmountRatio"] = voice_table["refillAmount"].apply(lambda x: int(x.replace("%", ""))*0.01 if pd.notna(x) else None)

In [21]:
voice_table.iloc[109]["refillRange"]

[]

In [22]:
with driver.session() as session:
    for _, row in voice_table.iterrows():
        # 속성값을 한글 컬럼명으로 변환
        properties = {}
        for col in eng_colnames:
            if type(row[col]) == list:
                properties[kor_cols_map[col]] = row[col]
            elif pd.isna(row[col]):
                # NaN 값을 'null'로 변환 -> 이렇게 해도 되는지 확인 필요 (neo4j에서는 NaN 또는 None을 속성값으로 넣을 수 없음)
                properties[kor_cols_map[col]] = 'null'
            else:
                properties[kor_cols_map[col]] = row[col]
        
        session.run(
            """
            MATCH (p:요금제 {고유ID: $product_id})
            MERGE (v:음성통화 {
                음성통화제공량: $음성통화제공량,
                영상및부가통화제공량: $영상및부가통화제공량,
                지정번호통화제공량: $지정번호통화제공량,
                리필비율한도: $리필비율한도,
                리필대상: $리필대상
            })
            MERGE (p)-[:제공]->(v)
            """,
            product_id=row['pmProductID'],
            음성통화제공량=properties['음성통화제공량'],
            영상및부가통화제공량=properties['영상및부가통화제공량'],
            지정번호통화제공량=properties['지정번호통화제공량'],
            리필비율한도=properties['리필비율한도'],
            리필대상=properties['리필대상']
        )

### SMS
- 문자메시지 제공량 관련 정보
- Label: 문자메시지
- 관계
  - 요금제 -[:제공]-> 문자메시지

In [23]:
# SMS 데이터 로드
sms_table = pd.read_csv(os.path.join(DATA_DIR, "SMS.csv"))
sms_table.head(1)

,pmProductID,includedText,textRange
0,PA00000001,99999,[]


In [24]:
# 컬럼명 매핑 정의
eng_colnames = [
    'includedText',
    'textRange'
]

kor_colnames = [
    '문자제공량',
    '문자대상'
]

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

In [25]:
# 그래프에 저장 -> 'textRange'는 빈 리스트만 존재해서 제외
with driver.session() as session:
    for _, row in sms_table.iterrows():
        properties = {kor_cols_map[col]: row[col] for col in eng_colnames}

        session.run(
            """
            MATCH (p:요금제 {고유ID: $product_id})
            MERGE (s:문자메시지 {
                문자제공량: $문자제공량
            })
            MERGE (p)-[:제공]->(s)
            """,
            product_id=row['pmProductID'],
            문자제공량=properties['문자제공량']
        )

### DATA
- 데이터 제공량 관련
- Label: 데이터용량
- 관계
  - 요금제 -[:제공]-> 데이터용량

In [26]:
data_table = pd.read_csv(os.path.join(DATA_DIR, "DATA.csv"))
data_table.head(1)

,pmProductID,includedData,includedDataForSharingAndTethering,includedMVoIP,appliedSpeed,seniorDataExceedAvailable,generalDataExceedAvailable,dataRefillAmount,dataRefillCouponGiftingAvailability,maximumShareAmount,dataGiftReceivingAvailability
0,PA00000001,15.0,15.0,15.0,1.0,N,N,15.0,Y,2GB,Y


In [27]:
# 컬럼명 매핑 정의
eng_colnames = [
    'includedData', 'includedDataForSharingAndTethering',
    'includedMVoIP', 'appliedSpeed', 'seniorDataExceedAvailable',
    'generalDataExceedAvailable', 'dataRefillAmount',
    'dataRefillCouponGiftingAvailability', 'maximumShareAmount',
    'dataGiftReceivingAvailability'
]

kor_colnames = [
    '기본제공데이터용량', '기본제공데이터중공유가능용량',
    '기본제공데이터중mvoip용량', '데이터소진후데이터제공속도', '시니어대상데이터소진후최대금액및속도제한적용',
    '데이터소진후최대금액및속도제한적용', '데이터리필가능용량',
    '데이터리필쿠폰선물가능여부', '최대데이터선물가능용량',
    '데이터선물받기가능여부'
]

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

In [28]:
data_table["maximumShareAmount"] = data_table["maximumShareAmount"].apply(lambda x: float(x.replace("GB", "")) if pd.notna(x) else None)

In [29]:
# data_table에서 NaN을 칼럼의 데이터타입에 맞춰 임의의 값으로 치환
data_table["includedData"] = data_table["includedData"].fillna(0.0)
data_table["includedDataForSharingAndTethering"] = data_table["includedDataForSharingAndTethering"].fillna(0.0)
data_table["includedMVoIP"] = data_table["includedMVoIP"].fillna(0.0)
data_table["appliedSpeed"] = data_table["appliedSpeed"].fillna(0.0)
data_table["seniorDataExceedAvailable"] = data_table["seniorDataExceedAvailable"].fillna("null")
data_table["generalDataExceedAvailable"] = data_table["generalDataExceedAvailable"].fillna("null")
data_table["dataRefillAmount"] = data_table["dataRefillAmount"].fillna(0.0)
data_table["dataRefillCouponGiftingAvailability"] = data_table["dataRefillCouponGiftingAvailability"].fillna("null")
data_table["maximumShareAmount"] = data_table["maximumShareAmount"].fillna(0.0)
data_table["dataGiftReceivingAvailability"] = data_table["dataGiftReceivingAvailability"].fillna("null")

In [30]:
# 그래프에 저장
with driver.session() as session:
    for _, row in data_table.iterrows():
        properties = {kor_cols_map[col]: row[col] for col in eng_colnames}

        session.run(
            """
            MATCH (p:요금제 {고유ID: $product_id})
            MERGE (d:데이터용량 {
            기본제공데이터용량: $기본제공데이터용량,
            기본제공데이터중공유가능용량: $기본제공데이터중공유가능용량,
            기본제공데이터중mvoip용량: $기본제공데이터중mvoip용량,
            데이터소진후데이터제공속도: $데이터소진후데이터제공속도,
            시니어대상데이터소진후최대금액및속도제한적용: $시니어대상데이터소진후최대금액및속도제한적용,
            데이터소진후최대금액및속도제한적용: $데이터소진후최대금액및속도제한적용,
            데이터리필가능용량: $데이터리필가능용량,
            데이터리필쿠폰선물가능여부: $데이터리필쿠폰선물가능여부,
            최대데이터선물가능용량: $최대데이터선물가능용량,
            데이터선물받기가능여부: $데이터선물받기가능여부
            })
            MERGE (p)-[:제공]->(d)
            """,
            product_id=row['pmProductID'],
            기본제공데이터용량=properties['기본제공데이터용량'],
            기본제공데이터중공유가능용량=properties['기본제공데이터중공유가능용량'],
            기본제공데이터중mvoip용량=properties['기본제공데이터중mvoip용량'],
            데이터소진후데이터제공속도=properties['데이터소진후데이터제공속도'],
            시니어대상데이터소진후최대금액및속도제한적용=properties['시니어대상데이터소진후최대금액및속도제한적용'],
            데이터소진후최대금액및속도제한적용=properties['데이터소진후최대금액및속도제한적용'],
            데이터리필가능용량=properties['데이터리필가능용량'],
            데이터리필쿠폰선물가능여부=properties['데이터리필쿠폰선물가능여부'],
            최대데이터선물가능용량=properties['최대데이터선물가능용량'],
            데이터선물받기가능여부=properties['데이터선물받기가능여부'],
        )

### TOPUP
- 충전 관련
- Label: 충전서비스
- 관계
  - 요금제 -[:가능]-> 충전서비스

In [31]:
topup_table = pd.read_csv(os.path.join(DATA_DIR, "TOPUP.csv"))
topup_table.head(1)

,pmProductID,reChargeAvailability,minimumChargeAmount,maximumChargeAmount
0,PA00000001,N,NaN,NaN


In [32]:
# 컬럼명 매핑 정의
eng_colnames = [
    'reChargeAvailability', 'minimumChargeAmount', 'maximumChargeAmount'
]

kor_colnames = [
    '충전서비스대상여부', '최소충전금액', '최대충전금액'
]

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

In [33]:
# NaN 처리
topup_table["minimumChargeAmount"] = topup_table["minimumChargeAmount"].fillna(0.0)
topup_table["maximumChargeAmount"] = topup_table["maximumChargeAmount"].fillna(0.0)

In [34]:
# 그래프에 저장
with driver.session() as session:
    for _, row in topup_table.iterrows():
        properties = {kor_cols_map[col]: row[col] for col in eng_colnames}

        session.run(
            """
            MATCH (p:요금제 {고유ID: $product_id})
            MERGE (t:충전서비스 {
                충전서비스대상여부: $충전서비스대상여부,
                최소충전금액: $최소충전금액,
                최대충전금액: $최대충전금액
            })
            MERGE (p)-[:제공]->(t)
            """,
            product_id=row['pmProductID'],
            충전서비스대상여부=properties['충전서비스대상여부'],
            최소충전금액=properties['최소충전금액'],
            최대충전금액=properties['최대충전금액']
        )

### CUSTOMERCONDITION
- 고객 가입 조건 데이터
- Label: 가입조건
- 관계
  - 요금제 -[:보유]-> 가입조건

In [35]:
condition_table = pd.read_csv(os.path.join(DATA_DIR, "CUSTOMERCONDITION.csv"))
condition_table.iloc[12:16]

,pmProductID,ageRule,customerTypeEligibility,customerTypeValueList,individualCustomerSubtypeEligibility,individualCustomerSubtypeValueList,directPlanOnboard,fixedPlanContractConcurrentSignupRestriction,tsupportFundOnboard,duplicateNameOnboardEligibility,duplicateNameOnboardGroupList,specialCustomerIsSoldier
12,PA00000013,[],NaN,NaN,NaN,NaN,Y,Y,Y,NaN,NaN,NaN
13,PA00000014,[],NaN,NaN,NaN,NaN,Y,Y,Y,NaN,NaN,NaN
14,PA00000015,"[{'eligibility': True, 'value': '연령 (월기준) 나이 이하 34'}]",True,['개인'],NaN,NaN,NaN,NaN,NaN,NaN,['TING_PRCPLN'],NaN
15,PA00000016,"[{'eligibility': True, 'value': '연령 (일기준) 나이 이상 65'}]",False,"['공공기관', '법인']",NaN,NaN,NaN,NaN,NaN,NaN,['ONESVC_SILVER_PROD'],NaN


In [36]:
# ageRule 파싱
ageRule_table = []
for _, row in condition_table.iterrows():
    ageRule = ast.literal_eval(row['ageRule'])
    if not ageRule:
        ageRule_table.append(["null", 0, "null", 999])
    else:
        # 초기값 (기본값)
        minAgeCriteria = "null"
        minAge = 0
        maxAgeCriteria = "null"
        maxAge = 999
        for rule in ageRule:
            if "이하" in rule["value"]:
                maxAge = int(regex.findall(r"\d+", rule["value"])[0])
                maxAgeCriteria = regex.findall(r"[일월]기준", rule["value"])[0]
            elif "이상" in rule["value"]:
                minAge = int(regex.findall(r"\d+", rule["value"])[0])
                minAgeCriteria = regex.findall(r"[일월]기준", rule["value"])[0]
        ageRule_table.append([minAgeCriteria, minAge, maxAgeCriteria, maxAge])

In [37]:
condition_table = pd.concat([condition_table, pd.DataFrame(ageRule_table, columns=["minAgeCriteria", "minAge", "maxAgeCriteria", "maxAge"])], axis=1)

In [38]:
condition_table.drop(columns=['ageRule'], inplace=True)

In [39]:
# 리스트형으로 변환
condition_table["customerTypeValueList"] = condition_table["customerTypeValueList"].apply(lambda x: type_cast(x))
condition_table["individualCustomerSubtypeValueList"] = condition_table["individualCustomerSubtypeValueList"].apply(lambda x: type_cast(x))
condition_table["duplicateNameOnboardGroupList"] = condition_table["duplicateNameOnboardGroupList"].apply(lambda x: type_cast(x))

In [40]:
# NaN 처리
condition_table["customerTypeEligibility"] = condition_table["customerTypeEligibility"].fillna("null")
condition_table["individualCustomerSubtypeEligibility"] = condition_table["individualCustomerSubtypeEligibility"].fillna("null")
condition_table["directPlanOnboard"] = condition_table["directPlanOnboard"].fillna("null")
condition_table["fixedPlanContractConcurrentSignupRestriction"] = condition_table["fixedPlanContractConcurrentSignupRestriction"].fillna("null")
condition_table["tsupportFundOnboard"] = condition_table["tsupportFundOnboard"].fillna("null")
condition_table["duplicateNameOnboardEligibility"] = condition_table["duplicateNameOnboardEligibility"].fillna("null")
condition_table["specialCustomerIsSoldier"] = condition_table["specialCustomerIsSoldier"].fillna("null")
condition_table["specialCustomerIsSoldier"] = condition_table["specialCustomerIsSoldier"].fillna("null")

In [41]:
condition_table.iloc[12:16]

,pmProductID,customerTypeEligibility,customerTypeValueList,individualCustomerSubtypeEligibility,individualCustomerSubtypeValueList,directPlanOnboard,fixedPlanContractConcurrentSignupRestriction,tsupportFundOnboard,duplicateNameOnboardEligibility,duplicateNameOnboardGroupList,specialCustomerIsSoldier,minAgeCriteria,minAge,maxAgeCriteria,maxAge
12,PA00000013,null,[],null,[],Y,Y,Y,null,[],null,null,0,null,999
13,PA00000014,null,[],null,[],Y,Y,Y,null,[],null,null,0,null,999
14,PA00000015,True,[개인],null,[],null,null,null,null,[TING_PRCPLN],null,null,0,월기준,34
15,PA00000016,False,"[공공기관, 법인]",null,[],null,null,null,null,[ONESVC_SILVER_PROD],null,일기준,65,null,999


In [42]:
eng_colnames = [
    'customerTypeEligibility', 'customerTypeValueList',
    'individualCustomerSubtypeEligibility',
    'individualCustomerSubtypeValueList', 'directPlanOnboard',
    'fixedPlanContractConcurrentSignupRestriction', 'tsupportFundOnboard',
    'duplicateNameOnboardEligibility', 'duplicateNameOnboardGroupList',
    'specialCustomerIsSoldier', 'minAgeCriteria', 'minAge',
    'maxAgeCriteria', 'maxAge'
]

kor_colnames = [
    '고객유형별가입가능여부', '고객유형목록',
    '개인고객세부유형별가입가능여부', 
    '개인고객세부유형목록', '다이렉트플랜가입가능여부',
    '선택약정동시가입가능여부', 'T지원금약정동시가입가능여부',
    '동일명의가입가능여부', '동일명의가입불가그룹목록',
    '군인전용요금제여부', '가입가능최소나이계산기준', '가입가능최소나이',
    '최대나이계산기준', '가입가능최대나이'
]

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

In [43]:
# 그래프에 저장
with driver.session() as session:
    for _, row in condition_table.iterrows():
        properties = {kor_cols_map[col]: row[col] for col in eng_colnames}

        session.run(
            """
            MATCH (p:요금제 {고유ID: $product_id})
            MERGE (c:가입조건 {
            고객유형별가입가능여부: $고객유형별가입가능여부,
            고객유형목록: $고객유형목록,
            개인고객세부유형별가입가능여부: $개인고객세부유형별가입가능여부,
            개인고객세부유형목록: $개인고객세부유형목록,
            다이렉트플랜가입가능여부: $다이렉트플랜가입가능여부,
            선택약정동시가입가능여부: $선택약정동시가입가능여부,
            T지원금약정동시가입가능여부: $T지원금약정동시가입가능여부,
            동일명의가입가능여부: $동일명의가입가능여부,
            동일명의가입불가그룹목록: $동일명의가입불가그룹목록,
            군인전용요금제여부: $군인전용요금제여부,
            가입가능최소나이계산기준: $가입가능최소나이계산기준,
            가입가능최소나이: $가입가능최소나이,
            최대나이계산기준: $최대나이계산기준,
            가입가능최대나이: $가입가능최대나이
            })
            MERGE (p)-[:보유]->(c)
            """,
            product_id=row['pmProductID'],
            고객유형별가입가능여부=properties['고객유형별가입가능여부'],
            고객유형목록=properties['고객유형목록'],
            개인고객세부유형별가입가능여부=properties['개인고객세부유형별가입가능여부'],
            개인고객세부유형목록=properties['개인고객세부유형목록'],
            다이렉트플랜가입가능여부=properties['다이렉트플랜가입가능여부'],
            선택약정동시가입가능여부=properties['선택약정동시가입가능여부'],
            T지원금약정동시가입가능여부=properties['T지원금약정동시가입가능여부'],
            동일명의가입가능여부=properties['동일명의가입가능여부'],
            동일명의가입불가그룹목록=properties['동일명의가입불가그룹목록'],
            군인전용요금제여부=properties['군인전용요금제여부'],
            가입가능최소나이계산기준=properties['가입가능최소나이계산기준'],
            가입가능최소나이=properties['가입가능최소나이'],
            최대나이계산기준=properties['최대나이계산기준'],
            가입가능최대나이=properties['가입가능최대나이']
        )

### product_group
- 요금제와 그룹간의 관계 정보
- Label: 요금제그룹
- 관계
  - 요금제 -[:속함]-> 요금제그룹

In [44]:
product_group_table = pd.read_csv(os.path.join(DATA_DIR, "product_group.csv"))
product_group_table.head(1)

,groupName,pmProductId,legacyProductId,productName
0,TING_PRCPLN,PA00000015,NA00006157,0플랜 라지


In [45]:
eng_colnames = ['groupName']
kor_colnames = ['그룹명']

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

In [46]:
# 그래프에 저장
with driver.session() as session:
    for _, row in product_group_table.iterrows():
        properties = {kor_cols_map[col]: row[col] for col in eng_colnames}

        # 그룹 노드 생성 및 요금제와 연결
        session.run(
            """
            MERGE (g:요금제그룹 {그룹명: $그룹명})
            WITH g
            MATCH (p:요금제 {고유ID: $pmProductId})
            MERGE (p)-[:속함]->(g)
            """,
            그룹명=properties['그룹명'],
            pmProductId=row['pmProductId']
        )

### BENEFITCONDITION
- 혜택 조건 정보 -> 구체적인 혜택 정보가 없어서 넣는 의미가 없어보여서 일단 스킵

### DEDUCTIBLE
- 장애인 고객 부가통화 제공량 확대 대상 여부
- Label: 장애인공제
- 관계
  - 요금제 -[:혜택]-> 장애인공제

In [47]:
deductitble_table = pd.read_csv(os.path.join(DATA_DIR, "DEDUCTIBLE.csv"))
deductitble_table.head(10)

,pmProductID,deductibilityForDisability,additionalOfferForDisabilities
0,PA00000001,N,NaN
1,PA00000002,Y,200분
2,PA00000003,N,NaN
3,PA00000004,N,NaN
4,PA00000005,N,NaN
5,PA00000006,N,NaN
6,PA00000007,N,NaN
7,PA00000008,N,NaN
8,PA00000009,N,NaN
9,PA00000010,N,NaN


In [48]:
# Normalize
deductitble_table["additionalOfferForDisabilities"] = deductitble_table["additionalOfferForDisabilities"].apply(lambda x: int(x.replace("분", "")) if pd.notna(x) else 0)

In [49]:
eng_colnames = ['additionalOfferForDisabilities']
kor_colnames = ['장애인부가통화추가제공량']

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

In [50]:
# 그래프에 저장
with driver.session() as session:
    for _, row in deductitble_table.iterrows():
        properties = {kor_cols_map[col]: row[col] for col in eng_colnames}

        session.run(
            """
            MATCH (p:요금제 {고유ID: $product_id})
            MERGE (d:장애인공제 {
                장애인부가통화추가제공량: $장애인부가통화추가제공량
            })
            MERGE (p)-[:혜택]->(d)
            """,
            product_id=row['pmProductID'],
            장애인부가통화추가제공량=properties['장애인부가통화추가제공량']
        )

### relation_db
- 상품과 혜택(부가서비스)간의 관계
- Label: 부가서비스
- 관계
  - 요금제 -[:??]-> 부가서비스

In [51]:
relation_db_table = pd.read_csv(os.path.join(DATA_DIR, "relation_db.csv"))

In [52]:
relation_db_table = relation_db_table[(relation_db_table["productId"].notna())&(relation_db_table["productId"] != "-")].copy()

In [53]:
relation_db_table.head()

,relationID,pmProductID,productId,productName,type
0,PA00000001_BA00000047,PA00000001,BA00000047,Wavve 2천원 할인,productBenefitConditions.allBenefitList
1,PA00000005_BA00000047,PA00000005,BA00000047,Wavve 2천원 할인,productBenefitConditions.allBenefitList
2,PA00000006_BA00000047,PA00000006,BA00000047,Wavve 2천원 할인,productBenefitConditions.allBenefitList
3,PA00000007_BA00000047,PA00000007,BA00000047,Wavve 2천원 할인,productBenefitConditions.allBenefitList
4,PA00000008_BA00000047,PA00000008,BA00000047,Wavve 2천원 할인,productBenefitConditions.allBenefitList


In [54]:
relation_db_table["type"].unique()

array(['productBenefitConditions.allBenefitList',
       'optionData.dataOptionProvidingMethod',
       'productRelation.signupConcurrentTermination.productList',
       'productRelation.signupPreTermination.productList',
       'productRelation.terminationConcurrentTermination.productList',
       'productRelation.terminationPreTermination.productList'],
      dtype=object)

In [55]:
relation_db_table.groupby("type").apply(lambda x: x.head(10)).reset_index(drop=True)

/tmp/ipykernel_92/759231149.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  relation_db_table.groupby("type").apply(lambda x: x.head(10)).reset_index(drop=True)


,relationID,pmProductID,productId,productName,type
0,PA00000059_NA00008243,PA00000059,NA00008243,베이직 플러스 데이터 충전 75GB,optionData.dataOptionProvidingMethod
1,PA00000059_NA00008242,PA00000059,NA00008242,베이직 플러스 데이터 충전 50GB,optionData.dataOptionProvidingMethod
2,PA00000059_NA00008241,PA00000059,NA00008241,베이직 플러스 데이터 충전 30GB,optionData.dataOptionProvidingMethod
3,PA00000059_NA00008240,PA00000059,NA00008240,베이직 플러스 데이터 충전 13GB,optionData.dataOptionProvidingMethod
4,PA00000070_NA00008710,PA00000070,NA00008710,컴팩트 데이터 충전 1GB,optionData.dataOptionProvidingMethod
5,PA00000081_NA00008705,PA00000081,NA00008705,0 청년 37 데이터 충전 1GB,optionData.dataOptionProvidingMethod
6,PA00000082_NA00008709,PA00000082,NA00008709,컴팩트플러스 데이터 충전 2GB,optionData.dataOptionProvidingMethod
7,PA00000084_NA00008707,PA00000084,NA00008707,슬림 데이터 충전 4GB,optionData.dataOptionProvidingMethod
8,PA00000085_NA00008708,PA00000085,NA00008708,베이직 데이터 충전 3GB,optionData.dataOptionProvidingMethod
9,PA00000086_NA00008703,PA00000086,NA00008703,0 청년 49 데이터 충전 4GB,optionData.dataOptionProvidingMethod


In [56]:
relation_type_translation = {
    'productBenefitConditions.allBenefitList': "할인", #부가서비스할인
    'optionData.dataOptionProvidingMethod': "데이터충전", #데이터충전혜택
    'productRelation.signupConcurrentTermination.productList': "가입동시해지",
    'productRelation.signupPreTermination.productList': "가입이전해지",
    'productRelation.terminationConcurrentTermination.productList': "해지동시해지",
    'productRelation.terminationPreTermination.productList': "해지이전해지"
}

In [57]:
eng_colnames = ["productId", "productName", "type"]
kor_colnames = ["부가서비스ID", "부가서비스명", "관계유형"]
kor_cols_map  = {x: y for x, y in zip(eng_colnames, kor_colnames)}

In [58]:
# 그래프에 저장
with driver.session() as session:
    for _, row in relation_db_table.iterrows():
        properties = {kor_cols_map[col]: row[col] for col in eng_colnames}
        relation_type = relation_type_translation[row['type']]

        if relation_type:
            session.run(
                f"""
                MATCH (p:요금제 {{고유ID: $product_id}})
                MERGE (b:부가서비스 {{부가서비스ID: $부가서비스ID, 부가서비스명: $부가서비스명}})
                MERGE (p)-[:{relation_type}]->(b)
                """,
                product_id=row['pmProductID'],
                부가서비스ID=properties['부가서비스ID'],
                부가서비스명=properties['부가서비스명']
            )


### 모든 boolean type을 string type으로 변경
- langchain_neo4j의 enhanced_schema를 사용할 때 boolean인 property 하나만 있는 경우 에러나는 버그가 있음

In [59]:
with driver.session() as session:
    session.run(
        f"""
        MATCH (n)
        UNWIND keys(n) AS key
        WITH n, key, n[key] AS value
        WHERE (value = true OR value = false)
        SET n[key] = CASE WHEN value = true THEN 'true' ELSE 'false' END
        """
    )

### 혜택 db
- 혜택정의_6월대상_20250611.xlsx 파일내 '요금제 혜택' 시트 6개를 하나로 통합한 파일 사용 

In [78]:
df = pd.read_csv('20250707_benefit_preprocessing.csv')

In [68]:
df.head(1)

,benefitInformation.pmBenefitCode,benefitInformation.benefitName,appliedDiscountProduct,product,pmProductId,legacyProductId,discountType,discountAmount,maxDiscountAmount,benefitType,benefitInformation.marketingKeyword
0,BA00000038,FLO 무료,"[{'product': 'FLO 앤 데이터', 'pmProductId': 'PB00000221', 'legacyProductId': 'NA00006520'}, {'product': 'FLO 앤 데이터 플러스', 'pmProductId': 'PB00000229', 'legacyProductId': 'NA00006599'}]",다이렉트5G 69(T 우주),PA00000010,NA00008100,정률할인,100%,7900원,요금 할인,"['플레이리스트', '음악앱', '취향', '플로', '음악', '이용권', 'Flo', '음악무료', '구.뮤직메이트', '노래', '뮤직', '음악어플', '뮤직앱', 'FLO앤데이터', '요금제할인', '플로할인', 'FLO앤데이터플러스', '요금제혜택']"


In [79]:
column_map = {
    'benefitInformation.pmBenefitCode': '혜택ID',
    'benefitInformation.benefitName': '혜택명',
    'appliedDiscountProduct':'혜택상품',
    'product': '요금제명',
    'pmProductId': '요금제ID',
    'legacyProductId': '요금제_SWINGID',
    'discountType': '할인유형',
    'discountAmount': '할인양',
    'maxDiscountAmount': '최대할인금액',
    'benefitType': '혜택유형',
    'benefitInformation.marketingKeyword': '마케팅키워드'
}


In [80]:
df = df.rename(columns=column_map)

In [81]:
def insert_benefit(tx, row):
    tx.run("""
    MERGE (b:혜택 {혜택ID: $혜택ID})
    SET
      b.혜택명 = $혜택명,
      b.혜택상품 = $혜택상품,
      b.요금제명 = $요금제명,
      b.요금제ID = $요금제ID,
      b.요금제_SWINGID = $요금제_SWINGID,
      b.할인유형 = $할인유형,
      b.할인양 = $할인양,
      b.최대할인금액 = $최대할인금액,
      b.혜택유형 = $혜택유형,
      b.마케팅키워드 = $마케팅키워드

    WITH b
    MATCH (p:요금제 {고유ID: $요금제ID})
    MERGE (p)-[:제공혜택]->(b)
    """, **row)

with driver.session() as session:
    for _, row in df.iterrows():
        # dict 변환 (NaN 처리)
        clean_row = {k: (v if pd.notnull(v) else None) for k, v in row.to_dict().items()}
        session.write_transaction(insert_benefit, clean_row)


/tmp/ipykernel_92/693942562.py:26: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(insert_benefit, clean_row)


In [60]:
driver.close()